# PIC-SURE on NHLBI BioData Catalyst®: from research question to analysis-ready data

## The question

> *Among COPD patients who have smoked, do carriers of a protein-altering (missense) variant in **SERPINA1**, the gene behind alpha-1 antitrypsin deficiency, show worse lung function?*

Answering this has traditionally meant separate data requests, reconciling study-specific variable definitions, and linking phenotype tables to genomic files by hand. Here it takes a few queries:

| Step | PIC-SURE capability | What it shows |
|---|---|---|
| **1** | **Cross-study search and DCC harmonized data** | One variable definition (smoking history) applied across every TOPMed study at once |
| **2** | **Study-specific (non-harmonized) data** | Drilling into the full dbGaP variable set of a single study (COPDGene) in the *same* query |
| **3** | **Genomic filtering** | Narrowing the cohort by variants in a gene |
| **4** | **Analysis-ready export** | A participant-level DataFrame, ready for statistics, without leaving the notebook |

## Setup

Install the PIC-SURE adapter and connect with a personal security token (BDC UI → **Prepare for Analysis** → **Copy**, saved as `token.txt` next to this notebook).

In [ ]:
%pip install picsure --upgrade

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import picsure

with open("token.txt") as f:
    my_token = f.read().strip()

session = picsure.connect(picsure.Platform.BDC_AUTHORIZED, my_token)

The whole demo is driven by these few settings. Change the gene or the variant consequence and re-run to ask a different question.

---
## Step 1 · One search across all of BDC, then one harmonized cohort across studies

A single dictionary search spans every study the user is authorized for. We can see this with the COPD search below.

In [ ]:
# Search for "COPD"
copd_search = session.searchDictionary("COPD")
copd_search.head()

Every study measures smoking differently: different questions, codings, and units. The **TOPMed DCC Harmonized data set** harmonizes phenotypic information across TOPMed cohorts. We use it to define our starting cohort: **ever-smokers, across all TOPMed studies.**

In [ ]:
# Utilize the dataset facets to get only results from the TOPMed DCC Harmonized data set
harmonized_facets = session.facets()
harmonized_facets.add("dataset_id", "DCC Harmonized data set")

# Search for "smoker"
smoking_vars = session.searchDictionary("smoker", facets=harmonized_facets)
smoking_vars

In [ ]:
# Get the PIC-SURE concept path and values of the variable of interest
ever_smoker = smoking_vars[smoking_vars.name == "ever_smoker_baseline_1"]
print(ever_smoker.conceptPath)
print(ever_smoker["values"])

Now, we can build a `clause` to filter to cigarette smokers using this variable. The `clause` can then be used in a `query` to get counts and retrieve participant data.

In [ ]:
# Build the clause to filter to cigarette smokers
ever_smoker_clause = picsure.buildClause(
    ever_smoker.conceptPath,
    picsure.PhenotypicFilterType.FILTER,
    categories="Current or former cigarette smoker"
)

In [ ]:
# Use the clause to get the count
ever_smoker_count = session.runQuery(
    picsure.buildQuery(phenotypicFilter=ever_smoker_clause),
    type=picsure.QueryType.COUNT
)
print(ever_smoker_count.value)

---
## Step 2 · Drill into a single study's own variables

For disease-specific detail we use the study's **original dbGaP variables**. COPDGene gives us a COPD diagnosis and spirometry (lung function). These go into the **same query** as the harmonized filter; PIC-SURE links participants across the two automatically.

In [ ]:
# Use facets to get results from COPDGene only
copdgene_facets = session.facets()
copdgene_facets.add("dataset_id", "phs000179")

# Search for COPD
copd_dx = session.searchDictionary("copd", facets=copdgene_facets)
copd_dx.head()

In [ ]:
# Save the "COPD: Have you ever had COPD" variable
copd_var = copd_dx.conceptPath[copd_dx.display == "COPD"]

# Create a clause to filter to those with COPD
copd_clause = picsure.buildClause(
    copd_var,
    picsure.PhenotypicFilterType.FILTER,
    categories="Yes"
)

In [ ]:
# Search for lung function (FEV1)
fev1_search = session.searchDictionary("FEV1", facets=copdgene_facets)
fev1_search.head()

In [ ]:
# Save FEV1 variable
fev1_var = fev1_search.conceptPath[fev1_search.display == "FEV1_FVC_pre"]

# Create a clause that selects 
fev1_clause = picsure.buildClause(
    fev1_var,
    picsure.PhenotypicFilterType.REQUIRE
)

PIC-SURE clauses can be combined in a query using AND and OR. Here, we will build a query using AND to combine the filters for smokers, COPD, and lung measurements.

In [ ]:
# Group phenotypic filters together with AND
copd_clausegroup = picsure.buildClauseGroup(
    [ever_smoker_clause, copd_clause, fev1_clause],
    operator=picsure.GroupOperator.AND
)

# Retrieve count of query
copd_count = session.runQuery(
    picsure.buildQuery(phenotypicFilter=copd_clausegroup), 
    type=picsure.QueryType.COUNT)

print(copd_count.value)

---
## Step 3 · Add a genomic filter

PIC-SURE indexes variant calls alongside phenotypes, so a gene-level filter is one line, with no VCFs to download or align.

To compare like with like, we first restrict to participants who **have been sequenced** (they have a TOPMed sample ID), then add the variant filter.

In [ ]:
# First, add filter to those with genomic information (have Sample IDs)
# Use facets to filter to COPDGene TOPMed study
wgs_facets = session.facets()
wgs_facets.add("dataset_id", "phs000951")

# Search for sample IDs
sample_ids = session.searchDictionary("sample id", facets=wgs_facets)
sample_paths = list(sample_ids.conceptPath[sample_ids.display == "SAMPLE_ID"])

In [ ]:
# Create a clause to select participants with at least one sample ID
# Multiple paths in one REQUIRE clause = "has a sample ID in any of these"
sequenced_clause = picsure.buildClause(sample_paths, picsure.PhenotypicFilterType.REQUIRE)

# Add this to the existing clause group
sequenced_clausegroup = picsure.buildClauseGroup(
    [copd_clausegroup, sequenced_clause], 
    operator=picsure.GroupOperator.AND
)

# Retrieve count for number of participants that have genetic info
sequenced_count = session.runQuery(
    picsure.buildQuery(phenotypicFilter=sequenced_clausegroup),
    type=picsure.QueryType.COUNT
)
print(sequenced_count.value)

Now, let's retrieve the number of participants with a missense variant of the *SERPINA1* gene.

In [ ]:
# Create variant filters - gene name and missense_variant
variant_filters = [
    picsure.buildGenomicFilter(picsure.GenomicFilterKey.GENE_WITH_VARIANT, values="SERPINA1"),
    picsure.buildGenomicFilter(picsure.GenomicFilterKey.VARIANT_CONSEQUENCE_CALCULATED, values="missense_variant"),
]

# Add genomic filter to phenotypic filter
carrier_query = picsure.buildQuery(phenotypicFilter=sequenced_clausegroup, genomicFilters=variant_filters)

# Retrieve count
carrier_count = session.runQuery(carrier_query, type=picsure.QueryType.COUNT)
print(carrier_count.value)

### The cohort, step by step

Every bar below is a live count returned by PIC-SURE.

In [ ]:
labels = ["Ever-smokers\n(all TOPMed,\nharmonized)", "+ COPD with\nspirometry\n(COPDGene)", "+ Sequenced\n(TOPMed WGS)", "+ SERPINA1\nmissense variant\n(genomic filter)"]
values = [ever_smoker_count.value, copd_count.value, sequenced_count.value, carrier_count.value]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color="#2a78d6", width=0.6)
ax.bar_label(bars, padding=3)  # value printed above each bar
ax.set_title("Cohort counts", loc="left")
ax.set_ylabel("Participant count")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 4 · Export analysis-ready data

Now we retrieve participant-level data: one row per participant, the variables we asked for as columns. We pull the sequenced COPD cohort and flag who carries a variant.

> ⚠️ **The cells below retrieve participant-level data.** Run them only in an environment approved under your Data Use Agreement, and do not display individual rows when presenting.

In [ ]:
# Run query to retrieve participant data for analysis
# Full cohort of those with genomic data sequenced
cohort_df = session.runQuery(
    picsure.buildQuery(phenotypicFilter=sequenced_clausegroup), 
    type=picsure.QueryType.PARTICIPANT
)

# Cohort of those with missense variant
carrier_ids = set(session.runQuery(carrier_query,type=picsure.QueryType.PARTICIPANT)["patient_id"])

In [ ]:
# Exploratory table for lung measurement across carriers and non-carriers
analysis = pd.DataFrame({
    "carrier": cohort_df["patient_id"].isin(carrier_ids).map({True: "SERPINA1 missense variant carrier", False: "Non-carrier"}),
    # A cell can hold repeated measurements separated by tabs; use the first
    "fev1": pd.to_numeric(cohort_df[fev1_var.iloc[0]].astype(str).str.split("\t").str[0], errors="coerce"),
}).dropna()

print(analysis.groupby("carrier")["fev1"].describe())

In [ ]:
# Box plot comparing lung function measurements between carriers and non-carriers
groups = ["Non-carrier", "SERPINA1 missense variant carrier"]
colors = ["#2a78d6", "#eb6834"]
data = [analysis.loc[analysis.carrier == g, "fev1"] for g in groups]

fig, ax = plt.subplots(figsize=(7, 4.5))
box = ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=False,
                 medianprops={"color": "#1a1a19", "linewidth": 2})
for patch, c in zip(box["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.35)
    patch.set_edgecolor(c)

ax.set_xticks([1, 2], [f"{g}\n(n = {len(d):,})" for g, d in zip(groups, data)])
ax.set_ylabel("FEV1")
ax.set_title(f"Lung function in COPDGene ever-smokers with COPD,\nby SERPINA1 missense variant status", loc="left", fontsize=11)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()